# CLIP + SimCLR Image Retrieval Training

Complete training notebook for:
- **CLIP**: Text-to-Image Retrieval (search images by text description)
- **SimCLR**: Image-to-Image Retrieval (find similar images)

## Key Optimisations in this version:
✅ BERT weights frozen — pre-computed once before training (eliminates 95% of compute)  
✅ Pre-tokenisation at dataset init — no tokenisation during training  
✅ Larger batch size (128) — better GPU utilisation  
✅ Both CLIP and SimCLR train on the same 30,000 COCO images  
✅ SimCLR uses the same image directory as CLIP  
✅ FAISS GPU-accelerated similarity search  

## Expected Results:
| Model | Metric | Expected Value |
|-------|--------|---------------|
| CLIP | Loss after epoch 1 | 4.0 – 5.0 |
| CLIP | Loss after epoch 10 | 1.5 – 2.5 |
| CLIP | Time per epoch | ~15–25 min (T4 GPU) |
| SimCLR | Loss after epoch 1 | 8.0 – 12.0 |
| SimCLR | Loss after epoch 10 | 2.0 – 4.0 |
| SimCLR | Time per epoch | ~8–15 min (T4 GPU) |
| Total training (both) | Wall clock | ~4–6 hours |

## Runtime: Google Colab with GPU (required)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# CELL 1 — Mount Google Drive

import os, sys
COLAB = True

project_dir = '/content/drive/MyDrive/Image-Retrieval-Model'
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

print("✓ Google Drive mounted")
print(f"✓ Working directory: {os.getcwd()}")


✓ Google Drive mounted
✓ Working directory: /content/drive/MyDrive/Image-Retrieval-Model


In [4]:
# CELL 2 — Clear stale module cache (run after any kernel restart)

import sys, gc, importlib

gc.collect()
stale = [m for m in list(sys.modules.keys())
         if 'torch' in m.lower() or 'tensorflow' in m.lower()]
for m in stale:
    sys.modules.pop(m, None)
importlib.invalidate_caches()
gc.collect()

print(f"✓ Cleared {len(stale)} stale modules")
print("  Proceed to install cell")


✓ Cleared 1 stale modules
  Proceed to install cell


In [5]:
# CELL 3 — Install / verify dependencies

import subprocess, sys

def pip(cmd):
    subprocess.check_call([sys.executable, '-m', 'pip'] + cmd.split(),
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Installing dependencies...")
pip('install -q --upgrade pip setuptools wheel')
pip('install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118')
pip('install -q transformers pillow tqdm numpy requests faiss-cpu scikit-learn')
print("✓ Dependencies installed")

# Quick verify
import torch, faiss
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")
print(f"  FAISS    : {faiss.get_num_gpus()} GPU(s) available")


Installing dependencies...
✓ Dependencies installed
  PyTorch  : 2.10.0+cu128
  CUDA     : True
  FAISS    : 0 GPU(s) available


In [6]:
# CELL 4 — Imports

import os, json, gc, shutil, zipfile, warnings
import numpy as np
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from PIL import Image
from tqdm import tqdm
import faiss
from transformers import BertModel, BertTokenizer

warnings.filterwarnings('ignore')
print(f"✓ All imports OK — PyTorch {torch.__version__}")


✓ All imports OK — PyTorch 2.10.0+cu128


In [7]:
# CELL 5 — GPU check

print("=" * 60)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠  No GPU — training will be very slow.")
    print("   Runtime → Change runtime type → GPU")
print(f"  Device: {device}")
print("=" * 60)


✓ GPU : Tesla T4
  VRAM: 15.6 GB
  Device: cuda


In [8]:
# CELL 5b — GPU memory cleanup utility
# Call free_gpu_memory() after any training or extraction block

import gc, torch

def free_gpu_memory(*objects_to_del):
    """
    Delete model/tensor objects, run garbage collection, and empty the
    CUDA cache.  Pass any Python objects you want explicitly deleted.

    Usage:
        free_gpu_memory(model, optimizer, dataloader)
    """
    for obj in objects_to_del:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        print(f"  GPU memory freed  |  allocated: {allocated:.2f} GB  |  reserved: {reserved:.2f} GB")
    else:
        print("  CPU mode — gc collected")

print("✓ free_gpu_memory() helper ready")


✓ free_gpu_memory() helper ready


In [10]:
# CELL 6 — Dataset paths (optimized to load only images in captions file)

import os, json, time

image_dir     = '/content/drive/MyDrive/coco/train2017'
captions_file = '/content/captions_train2017_sample_2000.json'

print(f"Image directory : {image_dir}")
print(f"Captions file   : {captions_file}")

assert os.path.exists(captions_file), f"❌ Captions file not found: {captions_file}"

# ── Load captions and extract image IDs ──────────────────────────────────────
with open(captions_file) as f:
    _d = json.load(f)

image_ids = set()
for a in _d['annotations']:
    image_ids.add(a['id'])

n_captions = sum(len(a['captions']) for a in _d['annotations'])

# Convert image IDs to filenames
valid_image_filenames = {f"{img_id:012d}.jpg" for img_id in image_ids}

# ── Check images by probing each file individually (avoids os.listdir on Drive)
print(f"\nChecking {len(valid_image_filenames):,} images individually (Drive-safe)...")

existing_images = set()
missing_images  = set()

for fname in valid_image_filenames:
    fpath = os.path.join(image_dir, fname)
    try:
        if os.path.isfile(fpath):
            existing_images.add(fname)
        else:
            missing_images.add(fname)
    except OSError:
        # retry once after brief pause (Drive hiccup)
        time.sleep(0.5)
        try:
            if os.path.isfile(fpath):
                existing_images.add(fname)
            else:
                missing_images.add(fname)
        except OSError:
            missing_images.add(fname)

n_images  = len(existing_images)
n_missing = len(missing_images)

valid_images_list = sorted(list(existing_images))
id_to_filename    = {img_id: f"{img_id:012d}.jpg" for img_id in image_ids}

del _d

print(f"✓ {n_images:,} images found (from {len(image_ids):,} in captions)")
print(f"✓ {n_captions:,} captions loaded")
if n_missing > 0:
    print(f"⚠  {n_missing:,} images missing from disk")

Image directory : /content/drive/MyDrive/coco/train2017
Captions file   : /content/captions_train2017_sample_2000.json

Checking 2,000 images individually (Drive-safe)...
✓ 1,999 images found (from 2,000 in captions)
✓ 10,002 captions loaded
⚠  1 images missing from disk


In [12]:
# CELL 7 — Loss functions

class CLIPLoss(nn.Module):
    """Symmetric cross-entropy loss for image-text matching."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / temperature))

    def forward(self, img_emb, txt_emb):
        img_emb = F.normalize(img_emb, p=2, dim=1)
        txt_emb = F.normalize(txt_emb, p=2, dim=1)
        logits  = img_emb @ txt_emb.T * self.logit_scale.exp()
        B       = img_emb.size(0)
        labels  = torch.arange(B, device=img_emb.device)
        loss    = (F.cross_entropy(logits, labels) +
                   F.cross_entropy(logits.T, labels)) / 2
        return loss


class SimCLRLoss(nn.Module):
    """NT-Xent loss for self-supervised image-image contrastive learning."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        B   = z1.size(0)
        z1  = F.normalize(z1, p=2, dim=1)
        z2  = F.normalize(z2, p=2, dim=1)
        z   = torch.cat([z1, z2], dim=0)           # (2B, D)
        sim = (z @ z.T) / self.temperature          # (2B, 2B)

        # mask out self-similarities
        mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask, -9e15)

        # positive indices: for row i its positive is at i+B (and vice versa)
        pos = torch.cat([torch.arange(B, 2*B, device=z.device),
                         torch.arange(0,  B,  device=z.device)])
        return F.cross_entropy(sim, pos)


print("✓ CLIPLoss defined")
print("✓ SimCLRLoss defined")


✓ CLIPLoss defined
✓ SimCLRLoss defined


In [13]:
# CELL 8 — Model definitions

# ── CLIP ─────────────────────────────────────────────────────────────────────

class CLIPImageEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        backbone = models.resnet50(pretrained=True)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(2048, 1024), nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(1024, embedding_dim)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


class CLIPTextEncoder(nn.Module):
    """
    BERT is used ONLY for pre-computing embeddings before training.
    During training only the tiny projection layer runs (Linear 768→512).
    """
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        for p in self.bert.parameters():   # freeze all BERT weights
            p.requires_grad = False
        self.projection = nn.Linear(768, embedding_dim)

    @torch.no_grad()
    def precompute(self, input_ids, attention_masks, device, batch_size=256):
        """Run BERT once on all captions → cache CLS vectors on CPU."""
        self.bert.eval().to(device)
        cls_list = []
        N = input_ids.shape[0]
        print(f"  Pre-computing BERT CLS embeddings for {N:,} captions...")
        for i in tqdm(range(0, N, batch_size), desc="  BERT pre-compute"):
            ids  = input_ids[i:i+batch_size].to(device)
            mask = attention_masks[i:i+batch_size].to(device)
            out  = self.bert(input_ids=ids, attention_mask=mask, return_dict=True)
            cls_list.append(out.last_hidden_state[:, 0, :].cpu())
        self.bert.cpu()
        torch.cuda.empty_cache()
        cache = torch.cat(cls_list, dim=0)   # (N, 768)  on CPU
        print(f"  ✓ Cache shape: {cache.shape}")
        return cache

    def forward_from_cache(self, cls_vec):
        """Project pre-computed CLS vectors → embedding_dim."""
        return self.projection(cls_vec)

    def forward(self, input_ids, attention_mask=None):
        """Full forward pass (used only for inference / query encoding)."""
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                        return_dict=True)
        return self.projection(out.last_hidden_state[:, 0, :])


class CLIPModel(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(embedding_dim)
        self.text_encoder  = CLIPTextEncoder(embedding_dim)
        self.embedding_dim = embedding_dim

    def forward_image(self, images):
        return self.image_encoder(images)

    def forward_text_cached(self, cls_vec):
        return self.text_encoder.forward_from_cache(cls_vec)

    def forward_text(self, input_ids, attention_mask=None):
        return self.text_encoder(input_ids, attention_mask)

    def forward(self, images, input_ids, attention_mask=None):
        return self.forward_image(images), self.forward_text(input_ids, attention_mask)


# ── SimCLR ───────────────────────────────────────────────────────────────────

class SimCLRModel(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        backbone = models.resnet50(pretrained=True)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projection_head = nn.Sequential(
            nn.Linear(2048, 2048), nn.ReLU(inplace=True),
            nn.Linear(2048, embedding_dim)
        )
        self.embedding_dim = embedding_dim

    def forward(self, x):
        return self.projection_head(self.backbone(x))

    def get_features(self, x):
        return self.backbone(x)


print("✓ CLIPModel defined (BERT frozen, projection-only during training)")
print("✓ SimCLRModel defined")


✓ CLIPModel defined (BERT frozen, projection-only during training)
✓ SimCLRModel defined


In [14]:
# CELL 9 — BERT tokeniser + batch pre-tokenisation helper

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_text(text, max_length=77):
    """Tokenise a string or list of strings. Returns (input_ids, attention_mask)."""
    if isinstance(text, str):
        text = [text]
    enc = bert_tokenizer(text, max_length=max_length,
                         padding='max_length', truncation=True,
                         return_tensors='pt')
    return enc['input_ids'], enc['attention_mask']


def pretokenize_all(captions_list, max_length=77, batch_size=512):
    """
    Tokenise every caption once at dataset init.
    Returns input_ids (N, L) and attention_masks (N, L) as CPU tensors.
    """
    all_ids, all_masks = [], []
    for i in tqdm(range(0, len(captions_list), batch_size), desc="  Pre-tokenising"):
        enc = bert_tokenizer(
            captions_list[i:i+batch_size],
            max_length=max_length, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        all_ids.append(enc['input_ids'])
        all_masks.append(enc['attention_mask'])
    ids   = torch.cat(all_ids,   dim=0)
    masks = torch.cat(all_masks, dim=0)
    print(f"  ✓ Pre-tokenised {ids.shape[0]:,} captions → shape {tuple(ids.shape)}")
    return ids, masks


print("✓ Tokeniser ready")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Tokeniser ready


In [15]:
# CELL 10 — Dataset classes

class CLIPDataset(Dataset):
    """
    Loads image-caption pairs from COCO-style JSON.
    Supported format  (your file):
        {"annotations": [{"id": 336354, "captions": ["...", ...]}, ...]}
    Pre-tokenises ALL captions at __init__ so __getitem__ is fast.
    """
    def __init__(self, image_dir, captions_file, transform=None, max_samples=None):
        self.image_dir = image_dir
        self.samples   = []   # list of {'image_id', 'caption'}
        self.id2file   = {}   # image_id -> filename

        with open(captions_file) as f:
            data = json.load(f)

        anns = data.get('annotations', [])
        if not anns:
            raise ValueError("No 'annotations' key found in captions JSON.")

        first = anns[0]

        if 'id' in first and 'captions' in first:
            # YOUR format: {id, captions:[...]}
            for ann in anns:
                img_id   = ann['id']
                filename = f"{img_id:012d}.jpg"
                self.id2file[img_id] = filename
                for cap in ann['captions']:
                    self.samples.append({'image_id': img_id, 'caption': cap})

        elif 'image_id' in first and 'caption' in first:
            # Standard COCO format
            if 'images' in data:
                self.id2file = {img['id']: img['file_name'] for img in data['images']}
            else:
                for ann in anns:
                    iid = ann['image_id']
                    self.id2file[iid] = f"{iid:012d}.jpg"
            self.samples = anns

        else:
            raise ValueError(f"Unknown annotation format. Keys: {list(first.keys())}")

        if max_samples:
            self.samples = self.samples[:max_samples]

        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                                  std=[0.229,0.224,0.225]),
        ])

        # Pre-tokenise every caption once
        print(f"  Loaded {len(self.samples):,} (image, caption) pairs from "
              f"{len(self.id2file):,} images")
        captions_text = [s['caption'] for s in self.samples]
        self.all_ids, self.all_masks = pretokenize_all(captions_text)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s      = self.samples[idx]
        img_id = s['image_id'] if 'image_id' in s else s.get('image_id')
        cap    = s['caption']  if 'caption'  in s else s.get('caption')

        fname  = self.id2file.get(img_id, f"{img_id:012d}.jpg")
        path   = os.path.join(self.image_dir, fname)
        try:
            image = Image.open(path).convert('RGB')
            image = self.transform(image)
        except Exception:
            image = torch.zeros(3, 224, 224)

        return image, self.all_ids[idx], self.all_masks[idx], fname, idx


class SimCLRDataset(Dataset):
    """
    Returns two differently-augmented views of each image.
    Accepts an explicit image list (filenames) so it trains on
    EXACTLY the same images referenced by the captions file.
    Falls back to scanning the directory if no list is provided.
    """
    def __init__(self, image_dir, image_files_list=None, num_images=None):
        self.image_dir = image_dir
        if image_files_list is not None:
            # Use the caller-supplied list (caption-filtered)
            self.image_files = sorted(image_files_list)
        else:
            self.image_files = sorted([
                f for f in os.listdir(image_dir)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))
            ])
        if num_images:
            self.image_files = self.image_files[:num_images]
        if not self.image_files:
            raise ValueError(f"No images found in {image_dir}")

        # Two stochastic augmentation pipelines
        _norm = transforms.Normalize(mean=[0.485,0.456,0.406],
                                      std=[0.229,0.224,0.225])
        self.t1 = transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(), _norm,
        ])
        self.t2 = transforms.Compose([
            transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.2, 0.05),
            transforms.RandomRotation(15),
            transforms.ToTensor(), _norm,
        ])
        print(f"  SimCLRDataset: {len(self.image_files):,} images from {image_dir}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_files[idx])
        try:
            img = Image.open(path).convert('RGB')
            return self.t1(img), self.t2(img), self.image_files[idx]
        except Exception:
            z = torch.zeros(3, 224, 224)
            return z, z, self.image_files[idx]


class ImageDataset(Dataset):
    """Plain image loader used for embedding extraction."""
    def __init__(self, image_dir):
        self.image_dir   = image_dir
        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg','.png','.jpeg'))
        ])
        self._norm = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                                  std=[0.229,0.224,0.225]),
        ])
        print(f"  ImageDataset: {len(self.image_files):,} images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_files[idx])
        try:
            img = Image.open(path).convert('RGB')
            return self._norm(img), self.image_files[idx]
        except Exception:
            return torch.zeros(3, 224, 224), self.image_files[idx]


print("✓ CLIPDataset, SimCLRDataset, ImageDataset defined")


✓ CLIPDataset, SimCLRDataset, ImageDataset defined


In [16]:
# CELL 11 — FAISS vector store

class FAISSVectorStore:
    def __init__(self, embedding_dim, use_gpu=True):
        self.dim      = embedding_dim
        self.metadata = []
        self.use_gpu  = use_gpu and faiss.get_num_gpus() > 0
        if self.use_gpu:
            res        = faiss.StandardGpuResources()
            self.index = faiss.GpuIndexFlatIP(res, embedding_dim)   # inner-product (cosine after norm)
            print("  ✓ FAISS GPU index")
        else:
            self.index = faiss.IndexFlatIP(embedding_dim)
            print("  ✓ FAISS CPU index")

    def add(self, embeddings: np.ndarray, names: list):
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings.astype(np.float32))
        self.metadata.extend(names)

    def search(self, query: np.ndarray, k=5):
        if query.ndim == 1:
            query = query.reshape(1, -1)
        q = query.copy().astype(np.float32)
        faiss.normalize_L2(q)
        scores, idxs = self.index.search(q, k)
        return [{'rank': r+1, 'name': self.metadata[i], 'score': float(s)}
                for r, (i, s) in enumerate(zip(idxs[0], scores[0])) if i >= 0]

    def save(self, index_path, meta_path):
        cpu_idx = faiss.index_gpu_to_cpu(self.index) if self.use_gpu else self.index
        faiss.write_index(cpu_idx, index_path)
        with open(meta_path, 'w') as f:
            json.dump({'dim': self.dim, 'n': self.index.ntotal,
                       'names': self.metadata}, f)
        print(f"  ✓ Saved {self.index.ntotal:,} vectors → {index_path}")

    def load(self, index_path, meta_path):
        idx = faiss.read_index(index_path)
        if self.use_gpu:
            res = faiss.StandardGpuResources()
            self.index = faiss.index_cpu_to_gpu(res, 0, idx)
        else:
            self.index = idx
        with open(meta_path) as f:
            d = json.load(f)
        self.metadata = d['names']

    @property
    def size(self):
        return self.index.ntotal


print("✓ FAISSVectorStore defined")


✓ FAISSVectorStore defined


In [17]:
# CELL 12 — Training configuration

config = {
    # ── shared ──────────────────────────────────────────────────────────────
    'embedding_dim' : 512,
    'batch_size'    : 128,   # larger batch → better GPU utilisation, fewer iters
    'learning_rate' : 1e-4,
    'weight_decay'  : 1e-5,
    'temperature'   : 0.07,
    'num_workers'   : 0,     # Colab requires 0; set to 2-4 on local machine
    'pin_memory'    : True,

    # ── per-model ────────────────────────────────────────────────────────────
    'clip_epochs'   : 5,    # ~15-25 min/epoch on T4
    'simclr_epochs' : 5,    # ~8-15  min/epoch on T4

    # ── BERT pre-compute batch ───────────────────────────────────────────────
    'bert_batch'    : 256,
}

print("Training configuration:")
for k, v in config.items():
    print(f"  {k:<20}: {v}")
print()
print("Expected timeline (T4 GPU):")
print("  BERT pre-compute   : ~3  min")
print("  CLIP  10 epochs    : ~3–4 hr")
print("  SimCLR 10 epochs   : ~2–3 hr")
print("  Total              : ~5–7 hr")

training_results = {}


Training configuration:
  embedding_dim       : 512
  batch_size          : 128
  learning_rate       : 0.0001
  weight_decay        : 1e-05
  temperature         : 0.07
  num_workers         : 0
  pin_memory          : True
  clip_epochs         : 5
  simclr_epochs       : 5
  bert_batch          : 256

Expected timeline (T4 GPU):
  BERT pre-compute   : ~3  min
  CLIP  10 epochs    : ~3–4 hr
  SimCLR 10 epochs   : ~2–3 hr
  Total              : ~5–7 hr


In [18]:
# CELL 13 — Image transforms

_norm = transforms.Normalize(mean=[0.485,0.456,0.406],
                              std=[0.229,0.224,0.225])

clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    _norm,
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    _norm,
])

print("✓ Transforms defined")


✓ Transforms defined


In [19]:
# CELL 14 — Train CLIP

print("=" * 60)
print("TRAINING CLIP")
print("=" * 60)

# ── Dataset ──────────────────────────────────────────────────────────────────
print("\nLoading CLIPDataset...")
clip_dataset = CLIPDataset(image_dir, captions_file, transform=clip_transform)

# ── Model + loss ─────────────────────────────────────────────────────────────
clip_model = CLIPModel(config['embedding_dim']).to(device)
clip_loss  = CLIPLoss(temperature=config['temperature']).to(device)

# ── Pre-compute BERT CLS vectors once ────────────────────────────────────────
print("\nPre-computing BERT embeddings (runs ONCE, then cached)...")
bert_cache = clip_model.text_encoder.precompute(
    clip_dataset.all_ids,
    clip_dataset.all_masks,
    device=device,
    batch_size=config['bert_batch']
)
# bert_cache: (N, 768) CPU tensor — indexed by sample index each batch

# ── DataLoader  (returns index so we can look up bert_cache) ─────────────────
clip_loader = DataLoader(
    clip_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    pin_memory=config['pin_memory'],
)

# ── Optimiser: only image encoder + text projection (BERT is frozen) ─────────
clip_optimizer = torch.optim.AdamW(
    list(clip_model.image_encoder.parameters()) +
    list(clip_model.text_encoder.projection.parameters()) +
    list(clip_loss.parameters()),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)
clip_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    clip_optimizer, T_max=config['clip_epochs']
)

# ── Training loop ─────────────────────────────────────────────────────────────
os.makedirs('checkpoints', exist_ok=True)
clip_losses  = []
clip_start   = datetime.now()

for epoch in range(config['clip_epochs']):
    clip_model.train()
    epoch_loss = 0.0

    pbar = tqdm(clip_loader,
                desc=f"CLIP Epoch {epoch+1}/{config['clip_epochs']}")

    for images, token_ids, attn_masks, img_names, indices in pbar:
        images = images.to(device)

        # Image branch — ResNet50 runs on GPU
        img_emb = clip_model.forward_image(images)

        # Text branch — only the tiny Linear projection, NO BERT
        cls_vec = bert_cache[indices].to(device)   # (B, 768)
        txt_emb = clip_model.forward_text_cached(cls_vec)

        loss = clip_loss(img_emb, txt_emb)

        clip_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(clip_model.parameters(), 1.0)
        clip_optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg = epoch_loss / len(clip_loader)
    clip_losses.append(avg)
    clip_scheduler.step()

    print(f"  Epoch {epoch+1:>2} | avg loss: {avg:.4f} | "
          f"lr: {clip_scheduler.get_last_lr()[0]:.2e}")

    torch.save(clip_model.state_dict(),
               f'checkpoints/clip_epoch{epoch+1}.pth')

clip_time = datetime.now() - clip_start
print(f"\n✓ CLIP training done in {clip_time}")
print(f"  Loss: {clip_losses[0]:.4f} → {clip_losses[-1]:.4f}")

# ── Free GPU memory after CLIP training ──────────────────────────────────────
print("\nFreeing GPU memory after CLIP training...")
free_gpu_memory(clip_loader, clip_optimizer, clip_scheduler)
# Keep clip_model in memory (needed for extraction), just clear intermediates
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"  GPU after CLIP: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


TRAINING CLIP

Loading CLIPDataset...
  Loaded 10,002 (image, caption) pairs from 2,000 images


  Pre-tokenising: 100%|██████████| 20/20 [00:04<00:00,  4.40it/s]


  ✓ Pre-tokenised 10,002 captions → shape (10002, 77)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 93.8MB/s]


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

  Epoch  3 | avg loss: 1.2333 | lr: 3.45e-05


CLIP Epoch 4/5: 100%|██████████| 79/79 [04:28<00:00,  3.40s/it, loss=0.1618]


  Epoch  4 | avg loss: 0.9454 | lr: 9.55e-06


CLIP Epoch 5/5: 100%|██████████| 79/79 [04:26<00:00,  3.38s/it, loss=0.1614]


  Epoch  5 | avg loss: 0.7923 | lr: 0.00e+00

✓ CLIP training done in 0:48:58.907498
  Loss: 3.1442 → 0.7923

Freeing GPU memory after CLIP training...
  GPU memory freed  |  allocated: 0.46 GB  |  reserved: 0.85 GB
  GPU after CLIP: 0.46 GB allocated


In [24]:
# CELL 15 — Train SimCLR  (same images as CLIP)

print("=" * 60)
print("TRAINING SimCLR")
print("=" * 60)

# ── Dataset — same image_dir as CLIP ─────────────────────────────────────────
print("\nLoading SimCLRDataset (same images as CLIP)...")
simclr_dataset = SimCLRDataset(image_dir, image_files_list=valid_images_list)  # same 2000 images as CLIP

simclr_loader = DataLoader(
    simclr_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers'],
    pin_memory=config['pin_memory'],
)

# ── Model + loss ──────────────────────────────────────────────────────────────
simclr_model = SimCLRModel(config['embedding_dim']).to(device)
simclr_loss  = SimCLRLoss(temperature=config['temperature'])

simclr_optimizer = torch.optim.AdamW(
    simclr_model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)
simclr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    simclr_optimizer, T_max=config['simclr_epochs']
)

# ── Training loop ─────────────────────────────────────────────────────────────
simclr_losses = []
simclr_start  = datetime.now()

for epoch in range(config['simclr_epochs']):
    simclr_model.train()
    epoch_loss = 0.0

    pbar = tqdm(simclr_loader,
                desc=f"SimCLR Epoch {epoch+1}/{config['simclr_epochs']}")

    for img1, img2, names in pbar:
        img1 = img1.to(device)
        img2 = img2.to(device)

        z1 = simclr_model(img1)
        z2 = simclr_model(img2)

        loss = simclr_loss(z1, z2)

        simclr_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(simclr_model.parameters(), 1.0)
        simclr_optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg = epoch_loss / len(simclr_loader)
    simclr_losses.append(avg)
    simclr_scheduler.step()

    print(f"  Epoch {epoch+1:>2} | avg loss: {avg:.4f} | "
          f"lr: {simclr_scheduler.get_last_lr()[0]:.2e}")

    torch.save(simclr_model.state_dict(),
               f'checkpoints/simclr_epoch{epoch+1}.pth')

simclr_time = datetime.now() - simclr_start
print(f"\n✓ SimCLR training done in {simclr_time}")
print(f"  Loss: {simclr_losses[0]:.4f} → {simclr_losses[-1]:.4f}")

# ── Free GPU memory after SimCLR training ────────────────────────────────────
print("\nFreeing GPU memory after SimCLR training...")
free_gpu_memory(simclr_loader, simclr_optimizer, simclr_scheduler)
# Keep simclr_model in memory (needed for extraction)
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"  GPU after SimCLR: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


TRAINING SimCLR

Loading SimCLRDataset (same images as CLIP)...
  SimCLRDataset: 1,999 images from /content/drive/MyDrive/coco/train2017


SimCLR Epoch 1/5:   0%|          | 0/16 [00:03<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 392.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 55.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 13.97 GiB is allocated by PyTorch, and 402.95 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# CELL 16 — Extract embeddings and build FAISS indexes

print("=" * 60)
print("EXTRACTING EMBEDDINGS & BUILDING FAISS INDEXES")
print("=" * 60)

eval_transform_local = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Use only the caption-filtered images for extraction too
class CaptionFilteredImageDataset(Dataset):
    """Plain image loader restricted to valid_images_list."""
    def __init__(self, image_dir, image_files, transform):
        self.image_dir   = image_dir
        self.image_files = sorted(image_files)
        self.transform   = transform
        print(f"  CaptionFilteredImageDataset: {len(self.image_files):,} images")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_files[idx])
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), self.image_files[idx]
        except Exception:
            return torch.zeros(3, 224, 224), self.image_files[idx]

filtered_dataset = CaptionFilteredImageDataset(
    image_dir, valid_images_list, eval_transform_local
)
eval_loader = DataLoader(
    filtered_dataset,
    batch_size=64, shuffle=False, num_workers=0, pin_memory=True,
)

# ── CLIP embeddings ───────────────────────────────────────────────────────────
print("\n[ CLIP ] Extracting image embeddings...")
clip_model.eval()
clip_embs, clip_names = [], []

with torch.no_grad():
    for images, names in tqdm(eval_loader, desc="  CLIP"):
        emb = clip_model.forward_image(images.to(device)).cpu().numpy()
        clip_embs.append(emb)
        clip_names.extend(names)

clip_embs = np.vstack(clip_embs)
print(f"  ✓ {clip_embs.shape[0]:,} CLIP embeddings, shape {clip_embs.shape}")

clip_store = FAISSVectorStore(config['embedding_dim'],
                              use_gpu=(faiss.get_num_gpus() > 0))
clip_store.add(clip_embs, clip_names)
print(f"  ✓ CLIP FAISS index: {clip_store.size:,} vectors")

# ── Free GPU after CLIP extraction ───────────────────────────────────────────
print("\nFreeing GPU memory after CLIP extraction...")
free_gpu_memory(clip_model)
del clip_embs          # large numpy array — free immediately
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"  GPU after CLIP extraction: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# ── SimCLR embeddings ─────────────────────────────────────────────────────────
print("\n[ SimCLR ] Extracting image embeddings...")
simclr_model.eval()
simclr_embs, simclr_names = [], []

with torch.no_grad():
    for images, names in tqdm(eval_loader, desc="  SimCLR"):
        emb = simclr_model(images.to(device)).cpu().numpy()
        simclr_embs.append(emb)
        simclr_names.extend(names)

simclr_embs = np.vstack(simclr_embs)
print(f"  ✓ {simclr_embs.shape[0]:,} SimCLR embeddings, shape {simclr_embs.shape}")

simclr_store = FAISSVectorStore(config['embedding_dim'],
                                use_gpu=(faiss.get_num_gpus() > 0))
simclr_store.add(simclr_embs, simclr_names)
print(f"  ✓ SimCLR FAISS index: {simclr_store.size:,} vectors")

# ── Free GPU after SimCLR extraction ─────────────────────────────────────────
print("\nFreeing GPU memory after SimCLR extraction...")
free_gpu_memory(simclr_model)
del simclr_embs
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"  GPU after SimCLR extraction: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# ── Keep eval_loader for retrieval test ──────────────────────────────────────
print("\n✓ Both indexes ready | GPU memory freed")


In [ ]:
# CELL 17 — Test retrieval (quick sanity check)

print("=" * 60)
print("RETRIEVAL TEST")
print("=" * 60)

clip_model.eval()

# ── CLIP: text → image ───────────────────────────────────────────────────────
print("\n[ CLIP ] Text-to-Image retrieval:")
test_queries = [
    "a dog playing in the park",
    "a person eating food",
    "cars on a busy street",
]

with torch.no_grad():
    for q in test_queries:
        ids, masks = tokenize_text(q)
        ids, masks = ids.to(device), masks.to(device)
        txt_emb = clip_model.forward_text(ids, masks).cpu().numpy()
        results = clip_store.search(txt_emb, k=3)
        print(f"\n  Query: \"{q}\"")
        for r in results:
            print(f"    #{r['rank']}  {r['name']}  (score: {r['score']:.4f})")

# ── SimCLR: image → image ────────────────────────────────────────────────────
print("\n[ SimCLR ] Image-to-Image retrieval:")
simclr_model.eval()
sample_names = simclr_names[:3]

_tf = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

with torch.no_grad():
    for fname in sample_names:
        path = os.path.join(image_dir, fname)
        img  = _tf(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
        emb  = simclr_model(img).cpu().numpy()
        results = simclr_store.search(emb, k=4)  # k=4 to skip rank-1 (self)
        print(f"\n  Query image: {fname}")
        for r in results[1:]:   # skip rank 1 (same image)
            print(f"    #{r['rank']-1}  {r['name']}  (score: {r['score']:.4f})")


In [ ]:
# CELL 18 — Save all models and FAISS indexes

print("=" * 60)
print("SAVING MODELS & INDEXES")
print("=" * 60)

for name, model, store, losses, elapsed in [
    ('clip',   clip_model,   clip_store,   clip_losses,   clip_time),
    ('simclr', simclr_model, simclr_store, simclr_losses, simclr_time),
]:
    out = f'trained_model_{name}'
    os.makedirs(out, exist_ok=True)

    # weights
    wpath = f'{out}/{name}_encoder.pth'
    torch.save(model.state_dict(), wpath)
    print(f"  ✓ {name.upper()} weights  → {wpath} "
          f"({os.path.getsize(wpath)/1e6:.1f} MB)")

    # FAISS
    store.save(f'{out}/embeddings.index', f'{out}/embeddings_meta.json')

    # config + summary
    summary = {
        'model': name, 'embedding_dim': config['embedding_dim'],
        'n_vectors': store.size, 'losses': losses,
        'final_loss': losses[-1], 'training_time': str(elapsed),
        'timestamp': datetime.now().isoformat(),
    }
    with open(f'{out}/summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  ✓ {name.upper()} summary  → {out}/summary.json")

print("\n✓ All saved")


In [ ]:
# CELL 19 — Zip and download

zip_path = 'Image-Retrieval-Models.zip'
print(f"Creating {zip_path}...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['trained_model_clip', 'trained_model_simclr']:
        for root, _, files in os.walk(folder):
            for fname in files:
                fp   = os.path.join(root, fname)
                arc  = os.path.relpath(fp, '.')
                zf.write(fp, arc)
                print(f"  + {arc}  ({os.path.getsize(fp)/1e6:.1f} MB)")

print(f"\n✓ Package size: {os.path.getsize(zip_path)/1e6:.1f} MB")

# save to Drive
drive_zip = f'/content/drive/MyDrive/Image-Retrieval-Models.zip'
shutil.copy(zip_path, drive_zip)
print(f"✓ Copied to Google Drive: {drive_zip}")

# browser download
try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Browser download started")
except Exception as e:
    print(f"  (auto-download skipped: {e})")


In [ ]:
# CELL 20 — Training summary

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)

print(f"\nCLIP")
print(f"  Epochs     : {config['clip_epochs']}")
print(f"  Loss curve : {' → '.join(f'{l:.3f}' for l in clip_losses)}")
print(f"  Time       : {clip_time}")

print(f"\nSimCLR")
print(f"  Epochs     : {config['simclr_epochs']}")
print(f"  Loss curve : {' → '.join(f'{l:.3f}' for l in simclr_losses)}")
print(f"  Time       : {simclr_time}")

print(f"\nBoth models trained on {n_images:,} images from:")
print(f"  {image_dir}")

print(f"\nExpected performance:")
print(f"  CLIP  — text query 'a dog' should return dog images in top-3")
print(f"  SimCLR — similar images (same scene/object) should cluster together")
print()
print("✓ Done. Download your models from Cell 19 or from Google Drive.")
